# Boston Housing Price Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してBostonデータセット（住宅価格データ）を探索し、高度な回帰モデルを構築します。

特徴量エンジニアリングとデータ標準化を活用した実践的なモデル構築を学びます。

In [1]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

Spark 依存関係が正常にロードされました


import $ivy.$
import $ivy.$

## 1. 環境設定とライブラリのインポート

In [2]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{SQLTransformer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder}
import org.apache.spark.ml.evaluation.RegressionEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("BostonExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/04 13:44:48 INFO SparkContext: Running Spark version 3.5.0
25/11/04 13:44:48 INFO SparkContext: OS info Windows 11, 10.0, amd64
25/11/04 13:44:48 INFO SparkContext: Java version 21.0.2
25/11/04 13:44:48 WARN Shell: Did not find winutils.exe: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
25/11/04 13:44:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 13:44:48 INFO ResourceUtils: ==============================================================
25/11/04 13:44:48 INFO ResourceUtils: No custom resources configured for spark.driver.
25/11/04 13:44:48 INFO ResourceUtils: ==============================================================
25/11/04 13:44:48 INFO SparkContext: Submitted application: BostonExploration
25/1

Spark Session created successfully!
Spark version: 3.5.0


import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{SQLTransformer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder}
import org.apache.spark.ml.evaluation.RegressionEvaluator
spark: SparkSession = org.apache.spark.sql.SparkSession@77942e66

## 2. データの読み込み

In [3]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/Boston.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

データ件数: 100

スキーマ:
root
 |-- CRIME: string (nullable = true)
 |-- ZN: double (nullable = true)
 |-- INDUS: double (nullable = true)
 |-- CHAS: integer (nullable = true)
 |-- NOX: double (nullable = true)
 |-- RM: double (nullable = true)
 |-- AGE: double (nullable = true)
 |-- DIS: double (nullable = true)
 |-- RAD: integer (nullable = true)
 |-- TAX: integer (nullable = true)
 |-- PTRATIO: double (nullable = true)
 |-- B: double (nullable = true)
 |-- LSTAT: double (nullable = true)
 |-- PRICE: double (nullable = true)



df: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 12 more fields]

## 3. データの概要確認

In [4]:
// 最初の10行を表示
df.show(10, truncate = false)

+--------+----+-----+----+-----+-----+----+------+----+---+-------+------+-----+-----+
|CRIME   |ZN  |INDUS|CHAS|NOX  |RM   |AGE |DIS   |RAD |TAX|PTRATIO|B     |LSTAT|PRICE|
+--------+----+-----+----+-----+-----+----+------+----+---+-------+------+-----+-----+
|high    |0.0 |18.1 |0   |0.718|3.561|87.9|1.6132|24  |666|20.2   |354.7 |7.12 |27.5 |
|low     |0.0 |8.14 |0   |0.538|5.95 |82.0|3.99  |4   |307|21.0   |232.6 |27.71|13.2 |
|very_low|82.5|2.03 |0   |0.415|6.162|38.4|6.27  |2   |348|14.7   |393.77|7.43 |24.1 |
|low     |0.0 |21.89|0   |0.624|6.151|97.9|1.6687|4   |437|21.2   |396.9 |18.46|17.8 |
|high    |0.0 |18.1 |0   |0.614|6.98 |67.6|2.5329|24  |666|20.2   |374.68|11.66|29.8 |
|low     |0.0 |6.2  |0   |0.507|6.086|61.5|3.6519|8   |307|17.4   |376.75|10.88|24.0 |
|very_low|22.0|5.86 |0   |0.431|6.438|8.9 |7.3967|7   |330|19.1   |377.07|3.59 |24.8 |
|very_low|0.0 |4.39 |0   |0.442|6.014|48.5|8.0136|3   |352|18.8   |385.64|10.53|17.5 |
|low     |0.0 |9.9  |0   |0.544|6.113|58.8|

In [5]:
// 統計情報
df.describe("RM", "LSTAT", "PTRATIO", "PRICE").show()

+-------+------------------+------------------+------------------+------------------+
|summary|                RM|             LSTAT|           PTRATIO|             PRICE|
+-------+------------------+------------------+------------------+------------------+
|  count|               100|               100|               100|               100|
|   mean| 6.235930000000001|11.826399999999994|18.517000000000007|23.457000000000004|
| stddev|0.7682871708343743| 6.830843377835949|1.9425098312349527| 9.570211095061612|
|    min|             3.561|              1.92|              13.0|               5.0|
|    max|             8.704|             30.59|              22.0|              50.0|
+-------+------------------+------------------+------------------+------------------+



In [6]:
// CRIME カテゴリの確認
df.groupBy("CRIME").count().orderBy("count").show()

+--------+-----+
|   CRIME|count|
+--------+-----+
|     low|   25|
|    high|   25|
|very_low|   50|
+--------+-----+



## 4. カテゴリカル変数のエンコーディング

In [7]:
// CRIME カテゴリカル変数のエンコーディング
val crimeIndexer = new StringIndexer()
  .setInputCol("CRIME")
  .setOutputCol("CRIME_index")

val crimeEncoder = new OneHotEncoder()
  .setInputCol("CRIME_index")
  .setOutputCol("CRIME_vec")

val encodePipeline = new Pipeline().setStages(Array(
  crimeIndexer, crimeEncoder
))

val encodedDF = encodePipeline.fit(df).transform(df)

println("カテゴリカル変数のエンコーディング完了")
encodedDF.select("CRIME", "CRIME_index", "CRIME_vec").show(5, truncate = false)

カテゴリカル変数のエンコーディング完了
+--------+-----------+-------------+
|CRIME   |CRIME_index|CRIME_vec    |
+--------+-----------+-------------+
|high    |1.0        |(2,[1],[1.0])|
|low     |2.0        |(2,[],[])    |
|very_low|0.0        |(2,[0],[1.0])|
|low     |2.0        |(2,[],[])    |
|high    |1.0        |(2,[1],[1.0])|
+--------+-----------+-------------+
only showing top 5 rows



crimeIndexer: StringIndexer = strIdx_840cce8cb963
crimeEncoder: OneHotEncoder = oneHotEncoder_5fe9781b721a
encodePipeline: Pipeline = pipeline_43dbb81d38d5
encodedDF: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 14 more fields]

## 5. 特徴量エンジニアリング

SQLTransformerを使用して、非線形な関係や交互作用を捉える特徴量を作成します。

In [8]:
// 特徴量エンジニアリング
val featureEngineering = new SQLTransformer().setStatement("""
  SELECT *,
    RM * RM as RM2,
    LSTAT * LSTAT as LSTAT2,
    PTRATIO * PTRATIO as PTRATIO2,
    RM * LSTAT as RM_LSTAT,
    RM * PTRATIO as RM_PTRATIO
  FROM __THIS__
""")

val engineeredDF = featureEngineering.transform(encodedDF)

println("特徴量エンジニアリング完了")
engineeredDF.select("RM", "RM2", "LSTAT", "LSTAT2", "RM_LSTAT").show(5)

特徴量エンジニアリング完了
+-----+------------------+-----+------------------+-----------------+
|   RM|               RM2|LSTAT|            LSTAT2|         RM_LSTAT|
+-----+------------------+-----+------------------+-----------------+
|3.561|         12.680721| 7.12|           50.6944|         25.35432|
| 5.95|           35.4025|27.71|          767.8441|         164.8745|
|6.162|         37.970244| 7.43|55.204899999999995|         45.78366|
|6.151|         37.834801|18.46|340.77160000000003|        113.54746|
| 6.98|48.720400000000005|11.66|          135.9556|81.38680000000001|
+-----+------------------+-----+------------------+-----------------+
only showing top 5 rows



featureEngineering: SQLTransformer = SQLTransformer: uid=sql_04ddeb8d1acd, statement=
  SELECT *,
    RM * RM as RM2,
    LSTAT * LSTAT as LSTAT2,
    PTRATIO * PTRATIO as PTRATIO2,
    RM * LSTAT as RM_LSTAT,
    RM * PTRATIO as RM_PTRATIO
  FROM __THIS__

engineeredDF: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 19 more fields]

## 6. データクリーニング

In [9]:
// 外れ値除去
val cleanedDF = engineeredDF
  .filter("PRICE < 50 AND PRICE > 0")
  .filter("RM > 0 AND LSTAT > 0")

println(s"データクリーニング:")
println(s"  元のデータ: ${engineeredDF.count()} 件")
println(s"  クリーニング後: ${cleanedDF.count()} 件")
println(s"  除去: ${engineeredDF.count() - cleanedDF.count()} 件")

データクリーニング:
  元のデータ: 100 件
  クリーニング後: 96 件
  除去: 4 件


cleanedDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [CRIME: string, ZN: double ... 19 more fields]

## 7. 特徴量の統合

In [10]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    // カテゴリカル特徴量（エンコード済み）
    "CRIME_vec",
    // 数値特徴量
    "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
    "RAD", "TAX", "PTRATIO", "B", "LSTAT",
    // エンジニアリングした特徴量
    "RM2", "LSTAT2", "PTRATIO2", "RM_LSTAT", "RM_PTRATIO"
  ))
  .setOutputCol("features")
  .setHandleInvalid("skip")

val assembledDF = assembler.transform(cleanedDF)

println(s"準備後のデータ件数: ${assembledDF.count()}")
assembledDF.select("features", "PRICE").show(5, truncate = false)

準備後のデータ件数: 94
+---------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                           |PRICE|
+---------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|[0.0,1.0,0.0,18.1,0.0,0.718,3.561,87.9,1.6132,24.0,666.0,20.2,354.7,7.12,12.680721,50.6944,408.03999999999996,25.35432,71.9322]                    |27.5 |
|[0.0,0.0,0.0,8.14,0.0,0.538,5.95,82.0,3.99,4.0,307.0,21.0,232.6,27.71,35.4025,767.8441,441.0,164.8745,124.95]                                      |13.2 |
|[1.0,0.0,82.5,2.03,0.0,0.415,6.162,38.4,6.27,2.0,348.0,14.7,393.77,7.43,37.970244,55.204899999999995,216.08999999999997,45.78366,90.58139999999999]|24.1 |
|[0.0,0.0,0.0,21.89,0.0,0.624,6.151,97.9,1.6687,4.

assembler: VectorAssembler = VectorAssembler: uid=vecAssembler_8aca5752808a, handleInvalid=skip, numInputCols=18
assembledDF: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 20 more fields]

## 8. データの標準化

StandardScalerを使用して、スケールの異なる特徴量を標準化します。

In [11]:
// StandardScalerによるデータ標準化
val scaler = new StandardScaler()
  .setInputCol("features")
  .setOutputCol("scaled_features")
  .setWithMean(true)   // 平均を0にする
  .setWithStd(true)    // 標準偏差を1にする

val scaledDF = scaler.fit(assembledDF).transform(assembledDF)

println("データ標準化完了")
scaledDF.select("features", "scaled_features").show(3, truncate = false)

データ標準化完了
+---------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                           |scaled_features                                                                                                                                                                                                                                                                                                             

scaler: StandardScaler = stdScal_1e3a406f28f2
scaledDF: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 21 more fields]

## 9. データの分割

In [12]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = scaledDF.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

訓練データ: 59 件
テストデータ: 35 件


trainData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [CRIME: string, ZN: double ... 21 more fields]
testData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [CRIME: string, ZN: double ... 21 more fields]

## 10. Linear Regressionモデルの訓練

In [13]:
// Linear Regressionモデルの作成
val lr = new LinearRegression()
  .setLabelCol("PRICE")
  .setFeaturesCol("scaled_features")
  .setMaxIter(100)
  .setRegParam(0.1)        // L2正則化
  .setElasticNetParam(0.0)  // 0=Ridge, 1=Lasso

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

モデルを訓練中...
訓練完了！


lr: LinearRegression = linReg_91459cbdcadf
pipeline: Pipeline = pipeline_a63b62013c7f
model: PipelineModel = pipeline_a63b62013c7f

## 11. モデルの評価

In [14]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new RegressionEvaluator()
  .setLabelCol("PRICE")
  .setPredictionCol("prediction")

val r2 = evaluator.setMetricName("r2").evaluate(predictions)
val rmse = evaluator.setMetricName("rmse").evaluate(predictions)
val mae = evaluator.setMetricName("mae").evaluate(predictions)

println(f"R² Score: ${r2 * 100}%.2f%%")
println(f"RMSE: $rmse%.4f")
println(f"MAE: $mae%.4f")

R² Score: 79.86%
RMSE: 4.4521
MAE: 3.3228


predictions: org.apache.spark.sql.package.DataFrame = [CRIME: string, ZN: double ... 22 more fields]
evaluator: RegressionEvaluator = RegressionEvaluator: uid=regEval_57d4e3714489, metricName=mae, throughOrigin=false
r2: Double = 0.7985589944701172
rmse: Double = 4.452105391861201
mae: Double = 3.3228417278399003

## 12. 予測結果の確認

In [15]:
// 予測結果のサンプル表示
predictions.select(
  "CRIME", "RM", "LSTAT", "PTRATIO",
  "PRICE", "prediction"
).show(15, truncate = false)

+-----+-----+-----+-------+-----+------------------+
|CRIME|RM   |LSTAT|PTRATIO|PRICE|prediction        |
+-----+-----+-----+-------+-----+------------------+
|high |5.427|18.14|20.2   |13.8 |14.935889018096297|
|high |4.138|23.34|20.2   |11.9 |12.390482478824081|
|high |6.38 |23.69|20.2   |13.1 |15.289965799409346|
|high |5.304|26.64|20.2   |10.4 |8.596765157910054 |
|high |5.52 |24.56|20.2   |12.3 |12.665428194761558|
|high |6.525|18.13|20.2   |14.1 |13.113629424190956|
|high |3.561|7.12 |20.2   |27.5 |15.7929565450078  |
|high |6.545|5.29 |20.2   |21.9 |26.390063175824793|
|high |5.88 |12.03|14.7   |19.1 |25.225979874525045|
|high |5.468|26.42|14.7   |15.6 |17.225650125702806|
|low  |5.981|11.65|17.4   |24.3 |24.68794480104649 |
|low  |8.247|3.95 |17.4   |48.3 |41.460045051495925|
|low  |6.041|7.7  |19.6   |20.4 |26.39044167879281 |
|low  |5.95 |27.71|21.0   |13.2 |11.73325565735098 |
|low  |6.072|13.04|21.0   |14.5 |19.427251802059594|
+-----+-----+-----+-------+-----+-------------

In [16]:
// 実際の値と予測値の比較（誤差を計算）
import org.apache.spark.sql.functions._

val comparison = predictions.select(
  col("PRICE").as("actual"),
  col("prediction"),
  abs(col("PRICE") - col("prediction")).as("error"),
  (abs(col("PRICE") - col("prediction")) / col("PRICE") * 100).as("error_pct")
)

comparison.describe("actual", "prediction", "error", "error_pct").show()

+-------+------------------+------------------+-------------------+------------------+
|summary|            actual|        prediction|              error|         error_pct|
+-------+------------------+------------------+-------------------+------------------+
|  count|                35|                35|                 35|                35|
|   mean|22.060000000000002|22.184595248960722| 3.3228417278399003|15.281993417203475|
| stddev|10.064357610779092| 7.896826530496588|  3.006359849497461|14.729700283486602|
|    min|              10.4| 8.596765157910054|0.09956501003248164|0.4762434215471546|
|    max|              48.5|41.460045051495925|   11.7070434549922| 75.46652397323481|
+-------+------------------+------------------+-------------------+------------------+



import org.apache.spark.sql.functions._
comparison: org.apache.spark.sql.package.DataFrame = [actual: double, prediction: double ... 2 more fields]

## 13. モデルの係数確認

In [17]:
// Linear Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.regression.LinearRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()
println("訓練セットでの性能:")
println(s"RMSE: ${lrModel.summary.rootMeanSquaredError}")
println(s"R²: ${lrModel.summary.r2}")

モデルの係数:
Intercept: 22.3868173799322
Coefficients: [-1.886149848312207,1.1838302814134307,-0.16606707728443734,0.03828654041861703,0.3055823634759219,-0.8802190000548528,-0.31568548440585703,-1.7109290344851296,-1.5070620761121953,0.13373391352974193,-2.231404873157127,-0.45759711071585163,0.8626389214790336,0.8530928486823107,5.317043316992067,0.32585546040683055,0.5027894737265521,-3.88402709268213,-1.635669043030236]

訓練セットでの性能:
RMSE: 2.6006802652772616
R²: 0.8508851856160741


lrModel: org.apache.spark.ml.regression.LinearRegressionModel = LinearRegressionModel: uid=linReg_91459cbdcadf, numFeatures=19

## 14. 特徴量の重要度確認

In [18]:
// 係数の絶対値を特徴量の重要度として表示
val featureNames = Array("CRIME_vec") ++ 
                   Array("ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT") ++
                   Array("RM2", "LSTAT2", "PTRATIO2", "RM_LSTAT", "RM_PTRATIO")

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).take(10).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

特徴量の重要度（係数の絶対値）:
LSTAT2              : 5.3170
RM_PTRATIO          : -3.8840
PTRATIO             : -2.2314
CRIME_vec           : -1.8861
DIS                 : -1.7109
RAD                 : -1.5071
ZN                  : 1.1838
RM                  : -0.8802
LSTAT               : 0.8626
RM2                 : 0.8531


featureNames: Array[String] = Array(
  "CRIME_vec",
  "ZN",
  "INDUS",
  "CHAS",
  "NOX",
  "RM",
  "AGE",
  "DIS",
  "RAD",
  "TAX",
  "PTRATIO",
  "B",
  "LSTAT",
  "RM2",
  "LSTAT2",
  "PTRATIO2",
  "RM_LSTAT",
  "RM_PTRATIO"
)
coefficients: Array[Double] = Array(
  -1.886149848312207,
  1.1838302814134307,
  -0.16606707728443734,
  0.03828654041861703,
  0.3055823634759219,
  -0.8802190000548528,
  -0.31568548440585703,
  -1.7109290344851296,
  -1.5070620761121953,
  0.13373391352974193,
  -2.231404873157127,
  -0.45759711071585163,
  0.8626389214790336,
  0.8530928486823107,
  5.317043316992067,
  0.32585546040683055,
  0.5027894737265521,
  -3.88402709268213,
  -1.635669043030236
)

## 15. クリーンアップ

In [19]:
// SparkSessionの停止
// spark.stop()
println("完了！")

完了！
